# Individual Differences Analysis

Examines per-participant and per-model variation in performance:
- Individual accuracy across conditions
- Identify strong/weak performers
- Participant×Condition interactions
- Performance heterogeneity within domains

**Prerequisites:** Run `Data-Preparation.ipynb` first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from src.data_loaders import load_human_master, load_model_master
from src.config import get_paths

paths = get_paths()
sns.set_style("whitegrid")

human_master = load_human_master()
model_master = load_model_master()

print(f"✓ Humans: {human_master['participantID'].nunique()} participants")
print(f"✓ Models: {model_master['participantID'].nunique()} models")

## 1. Individual Performance Summary

In [ ]:
def compute_individual_metrics(df, domain_name):
    """Compute accuracy and SDT metrics per participant."""
    rows = []
    
    for pid in df['participantID'].unique():
        pdf = df[df['participantID'] == pid]
        
        for cond in df['condition'].unique():
            cpdf = pdf[pdf['condition'] == cond]
            if len(cpdf) == 0:
                continue
            
            # Compute metrics
            accuracy = (cpdf['decision'] == cpdf['TP']).mean()
            hits = np.sum((cpdf['decision'] == 1) & (cpdf['TP'] == 1))
            fas = np.sum((cpdf['decision'] == 1) & (cpdf['TP'] == 0))
            n_signal = np.sum(cpdf['TP'] == 1)
            n_noise = np.sum(cpdf['TP'] == 0)
            
            H = (hits + 0.5) / (n_signal + 1) if n_signal > 0 else np.nan
            F = (fas + 0.5) / (n_noise + 1) if n_noise > 0 else np.nan
            
            if np.isfinite(H) and np.isfinite(F):
                zH = norm.ppf(H)
                zF = norm.ppf(F)
                dprime = zH - zF
                criterion = -0.5 * (zH + zF)
            else:
                dprime = np.nan
                criterion = np.nan
            
            rows.append({
                "domain": domain_name,
                "participantID": pid,
                "condition": cond,
                "n_trials": len(cpdf),
                "accuracy": accuracy,
                "hit_rate": H,
                "fa_rate": F,
                "dprime": dprime,
                "criterion": criterion
            })
    
    return pd.DataFrame(rows)

human_indiv = compute_individual_metrics(human_master, "Human")
model_indiv = compute_individual_metrics(model_master, "Model")

indiv_all = pd.concat([human_indiv, model_indiv], ignore_index=True)

print("Human Individual Performance:")
display(human_indiv.sort_values("accuracy", ascending=False).head(10))

print("\nModel Individual Performance:")
display(model_indiv.sort_values("accuracy", ascending=False).head(15))

## 2. Performance Variability by Domain

In [ ]:
# Summary statistics
for domain in ["Human", "Model"]:
    df = indiv_all[indiv_all["domain"] == domain]
    print(f"\n{domain}:")
    print(f"  Accuracy: mean={df['accuracy'].mean():.3f}, std={df['accuracy'].std():.3f}")
    print(f"  d': mean={df['dprime'].mean():.3f}, std={df['dprime'].std():.3f}")
    print(f"  Min accuracy: {df['accuracy'].min():.3f}")
    print(f"  Max accuracy: {df['accuracy'].max():.3f}")

## 3. Participant × Condition Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, domain in zip(axes, ["Human", "Model"]):
    df = indiv_all[indiv_all["domain"] == domain]
    pivot = df.pivot(index="participantID", columns="condition", values="accuracy")
    pivot = pivot[["50_50", "80_20", "100_0"]]  # reorder columns
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0.5,
        vmax=1.0,
        ax=ax,
        cbar_kws={"label": "Accuracy"}
    )
    ax.set_title(f"{domain}: Accuracy by Participant × Condition", fontsize=12)
    ax.set_xlabel("Condition")
    ax.set_ylabel("Participant ID")

plt.tight_layout()
plt.savefig(paths.outputs_dir / "individual-differences-heatmap.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: individual-differences-heatmap.pdf")

## 4. Accuracy Distribution by Domain

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for domain in ["Human", "Model"]:
    df = indiv_all[indiv_all["domain"] == domain]
    ax.hist(
        df["accuracy"],
        bins=15,
        alpha=0.6,
        label=domain,
        edgecolor="black"
    )

ax.set_xlabel("Accuracy", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Distribution of Individual Accuracies", fontsize=12)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(paths.outputs_dir / "accuracy-distribution.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: accuracy-distribution.pdf")

## 5. d' (Discriminability) by Domain and Condition

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for domain in ["Human", "Model"]:
    df = indiv_all[indiv_all["domain"] == domain]
    
    means = df.groupby("condition")["dprime"].mean()
    stds = df.groupby("condition")["dprime"].std()
    
    x = np.arange(len(means))
    offset = 0.2 if domain == "Human" else -0.2
    
    ax.bar(
        x + offset,
        means,
        width=0.35,
        label=domain,
        alpha=0.7,
        yerr=stds,
        capsize=5,
        error_kw={"elinewidth": 2}
    )

ax.set_xticks(np.arange(3))
ax.set_xticklabels(["50_50", "80_20", "100_0"])
ax.set_xlabel("Condition", fontsize=12)
ax.set_ylabel("d' (Mean ± Std)", fontsize=12)
ax.set_title("Signal Detectability by Domain and Condition", fontsize=12)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(paths.outputs_dir / "dprime-by-condition.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: dprime-by-condition.pdf")

## 6. Outlier Detection

Identify strong and weak performers relative to domain mean.

In [ ]:
def find_outliers(df, domain_name, threshold=1.5):
    """Find accuracy outliers (>1.5 std from mean)."""
    domain_df = df[df["domain"] == domain_name]
    mean_acc = domain_df["accuracy"].mean()
    std_acc = domain_df["accuracy"].std()
    
    lower_bound = mean_acc - threshold * std_acc
    upper_bound = mean_acc + threshold * std_acc
    
    outliers_low = domain_df[domain_df["accuracy"] < lower_bound]
    outliers_high = domain_df[domain_df["accuracy"] > upper_bound]
    
    return outliers_low, outliers_high, mean_acc, std_acc

for domain in ["Human", "Model"]:
    low, high, mean_acc, std_acc = find_outliers(indiv_all, domain)
    
    print(f"\n{domain}:")
    print(f"  Mean accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"\n  Strong performers (>+1.5σ):")
    if len(high) > 0:
        for _, row in high.iterrows():
            print(f"    {row['participantID']:20s} {row['condition']:6s} acc={row['accuracy']:.3f}")
    else:
        print("    (none)")
    
    print(f"\n  Weak performers (<-1.5σ):")
    if len(low) > 0:
        for _, row in low.iterrows():
            print(f"    {row['participantID']:20s} {row['condition']:6s} acc={row['accuracy']:.3f}")
    else:
        print("    (none)")

## 7. Save Results

In [ ]:
indiv_all.to_csv(paths.outputs_dir / "individual-differences.csv", index=False)
print(f"✓ Saved: individual-differences.csv")